## **Part One**

### The data contains stock returns of small retailers

### An analyst claims there is **a bias in moment conditions of instrumental variables** such that **$Z^T$(Y - XB) =  $\sigma \begin{bmatrix} 1 \\ 1 \\ 1 \end{bmatrix}$**


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import GMM

In [2]:
input_table = pd.read_csv("/Users/lucasben/Documents/mba-business-analytics/courses/predictive-modelling/midterm-project/midterm_partone.csv")

In [3]:
model_iv = sm.OLS(input_table["Inventory Turnover"],input_table[["Constant","Current Ratio","Quick Ratio",\
                                                                 "Debt Asset Ratio"]]).fit()
endog_predict = model_iv.predict(input_table[["Constant","Current Ratio","Quick Ratio","Debt Asset Ratio"]])
input_table["Endogenous Param"] = endog_predict

In [4]:
model_2sls = sm.OLS(input_table["Stock Change"], input_table[["Constant","Endogenous Param",\
                                                              "Operating Profit","Interaction Effect",\
                                                             ]]).fit()
model_2sls.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           Stock Change   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     8.530
Date:                Mon, 10 Nov 2025   Prob (F-statistic):           1.27e-05
Time:                        10:36:10   Log-Likelihood:                -1186.5
No. Observations:                1696   AIC:                             2381.
Df Residuals:                    1692   BIC:                             2403.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Constant              -0.0176      0.020     -0.896      0.370      -0.056       0.021
Endogenous Param       0.0011      0.001      1.827      0.068   -7.76e-05       0.002
Operating Profit      -0.1201      0.028     -4.319      0.000      -0.175      -0.066
Interaction Effect     0.0014      0.000      3.621      0.000       0.001       0.002
==============================================================================
Omnibus:                      368.832   Durbin-Watson:                   2.243
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             3433.920
Skew:                           0.742   Prob(JB):                         0.00
Kurtosis:                       9.811   Cond. No.                         109.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [5]:
y  = np.array(input_table["Stock Change"])
X = np.array(input_table[["Inventory Turnover","Operating Profit","Interaction Effect"]])
Z = np.array(input_table[["Current Ratio","Quick Ratio","Debt Asset Ratio"]])


In [6]:
# calculating the average bias across the three instrumental variables
beta = np.array([0.1, 0.1, 0.1])
residuals = y - X.dot(beta)
Zt_res = Z.T.dot(residuals)
delta = Zt_res.mean()
delta


-4330.7898131218435

In [7]:
class gmm(GMM):
    def __init__(self, *args, delta=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.delta = delta 

    def momcond(self, params): # these are the conditions I am satisfying when estimating the parameters 
        p0, p1, p2, p3 = params # these are the coefficients I am estimating
        # p0 is the intercept 
        # p1 is inventory turnover
        # p2 is operating profit
        # p3 is the interaction between inventory turnover and operating profit
        endog = self.endog
        exog = self.exog
        inst = self.instrument   

        error0 = endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]
        error1 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * exog[:,1]
        error2 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * exog[:,2]
        error3 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * inst[:,0] - self.delta
        error4 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * inst[:,1] - self.delta
        error5 = (endog - p0 - p1 * exog[:,0] - p2 * exog[:,1] - p3 * exog[:,2]) * inst[:,2] - self.delta

        g = np.column_stack((error0, error1, error2, error3, error4, error5)) # this stacks all the error vectors into a matrix of moment conditions
        return g


beta0 = np.array([0.1, 0.1, 0.1, 0.1]) # the initial guess for the coefficients, GMM needs a starting point to begin its optimization 
res = gmm(endog = y, exog = X, instrument = Z, k_moms=6, k_params=4, delta=delta).fit(beta0)

res.summary()

/opt/anaconda3/lib/python3.12/site-packages/scipy/optimize/_optimize.py:1360: OptimizeWarning: Desired error not necessarily achieved due to precision loss.
  res = _minimize_bfgs(f, x0, args, fprime, callback=callback, **opts)


         Current function value: 24037394.087226
         Iterations: 15
         Function evaluations: 58
         Gradient evaluations: 47
Optimization terminated successfully.
         Current function value: 1.106415
         Iterations: 32
         Function evaluations: 45
         Gradient evaluations: 45
Optimization terminated successfully.
         Current function value: 0.569082
         Iterations: 65
         Function evaluations: 80
         Gradient evaluations: 80
Optimization terminated successfully.
         Current function value: 0.957930
         Iterations: 39
         Function evaluations: 51
         Gradient evaluations: 51
Optimization terminated successfully.
         Current function value: 1.968542
         Iterations: 81
         Function evaluations: 95
         Gradient evaluations: 95
Optimization terminated successfully.
         Current function value: 0.067861
         Iterations: 6
         Function evaluations: 26
         Gradient evaluations: 26


<class 'statsmodels.iolib.summary.Summary'>
"""
                                 gmm Results                                  
==============================================================================
Dep. Variable:                      y   Hansen J:                        121.0
Model:                            gmm   Prob (Hansen J):              5.43e-27
Method:                           GMM                                         
Date:                Mon, 10 Nov 2025                                         
Time:                        10:36:10                                         
No. Observations:                1696                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
p 0         4.146e+04   6112.720      6.782      0.000    2.95e+04    5.34e+04
p 1         -488.1581     92.180     -5.296      0.000    -668.827    -307.489
p 2        -1.353e+05   2.39e+04     -5.673      0.000   -1.82e+05   -8.86e+04
p 3         1153.9715    229.308      5.032      0.000     704.535    1603.408
==============================================================================
"""

## **Part Two**

In [9]:
df2 = pd.read_csv("/Users/lucasben/Documents/mba-business-analytics/courses/predictive-modelling/midterm-project/midterm_parttwo.csv")
df2.head()

,Years of Education after High School,Requested Credit Amount,Number of Dependents,Monthly Income,Monthly Expense,Marital Status,Credit Rating
0,1,Low,No dependent,Very low,Very low,Married,Positive
1,2,Low,No dependent,Very low,Very low,Single,Positive
2,1,Low,No dependent,Very low,Very low,Single,Positive
3,3,Low,No dependent,Very low,Very low,Married,Positive
4,3,Low,No dependent,Very low,Very low,Single,Negative


## **Data Understanding & Preprocessing**

In [ ]:
df2.isna().sum() # no missing values

Years of Education after High School    0
Requested Credit Amount                 0
Number of Dependents                    0
Monthly Income                          0
Monthly Expense                         0
Marital Status                          0
Credit Rating                           0
dtype: int64

In [34]:
df2.dtypes

Years of Education after High School     int64
Requested Credit Amount                 object
Number of Dependents                    object
Monthly Income                          object
Monthly Expense                         object
Marital Status                          object
Credit Rating                           object
dtype: object

In [28]:
df2["Requested Credit Amount"].value_counts(normalize=True) * 100

Requested Credit Amount
Low       67.343151
Medium    31.765871
High       0.890979
Name: proportion, dtype: float64

In [30]:
df2["Number of Dependents"].value_counts(normalize=True) * 100

Number of Dependents
Less than 2     44.746937
No dependent    44.635565
More than 2     10.617498
Name: proportion, dtype: float64

In [31]:
df2["Monthly Income"].value_counts(normalize=True) * 100

Monthly Income
Very low     39.388690
Low          35.082292
Moderate     17.572083
High          6.125480
Very High     1.831457
Name: proportion, dtype: float64

In [32]:
df2["Monthly Expense"].value_counts(normalize=True) * 100

Monthly Expense
Very low     77.255290
Low          18.302190
Moderate      2.920431
High          1.163222
Very high     0.358866
Name: proportion, dtype: float64

In [33]:
df2["Marital Status"].value_counts(normalize=True) * 100 

Marital Status
Married          72.713773
Single           25.491895
Not specified     1.794332
Name: proportion, dtype: float64

There is a class imbalance for the output variable. 85% of clients pay their debt in full.

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

cat_feats = ['Requested Credit Amount', 'Marital Status', 'Number of Dependents', 'Monthly Income', 'Monthly Expense']
num_feats = ['Years of Education after High School']

def combine_names(feature, category):
    return f"{feature}--{category}"

encoder = OneHotEncoder(
    min_frequency=0.06,
    handle_unknown='infrequent_if_exist',
    feature_name_combiner=combine_names
)

num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("encoder", encoder)
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_feats),
    ("cat", cat_pipeline, cat_feats)
])

lr_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1500, random_state=42))
])

In [36]:
# I want to build a logistic regression model that predicts credit rating

X, y = df2.drop(columns=['Credit Rating']), df2['Credit Rating']

y.value_counts(normalize=True)

Credit Rating
Positive    0.858186
Negative    0.141814
Name: proportion, dtype: float64

In [37]:
# training the model without accounting for class imbalance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [38]:
# training a binary logistic regression model

lr_pipeline.fit(X_train, y_train)

y_pred = lr_pipeline.predict(X_test)

In [39]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

confusion_matrix = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix, display_labels=lr.pipeline.classes_)
display.plot(cmap='Blues')

ValueError: pos_label=1 is not a valid label. It should be one of ['Negative', 'Positive']